# Notebook 03: Distribution Modeling

Chúng ta fit các **candidate distribution** và đánh giá khả năng nắm bắt cả hành vi **central behavior** lẫn đặc trưng **tail** của từng mô hình.

---

## Objectives

1. Fit ba họ **parametric distribution** — **Normal**, **Student-t**, và **Alpha-stable** — lên từng **dataset** bằng **MLE**.
2. So sánh các mô hình theo **AIC** và **BIC** để chọn mô hình tốt nhất có tính đến số **parameter**.
3. Đánh giá **goodness-of-fit** qua **KS test** và **tail quantile error** — phân biệt mô hình fit tốt toàn bộ và mô hình fit tốt ở **tail**.
4. Trực quan hóa chất lượng fit qua **PP plot**, **QQ plot**, và so sánh **fitted PDF** với **empirical density**.
5. Tính **bootstrap CI** cho các **parameter** quan trọng để định lượng **estimation uncertainty**.

---
## 0. Setup & Imports

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# Project modules
from src.distributions.fitter import (
    compare_distributions,
    best_distribution,
    fit_distribution,
)
from src.distributions import normal, student_t, stable
from src.evaluation.metrics import (
    quantile_error,
    tail_quantile_error,
    tail_probability_error,
    kolmogorov_smirnov_distance,
)
from src.evaluation.uncertainty import bootstrap_parameter_cis

warnings.filterwarnings("ignore")   # tắt scipy convergence warnings

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

COLORS = {
    "Gaussian":     "#2166AC",
    "Student-t(5)": "#4DAC26",
    "Student-t(3)": "#F1A340",
    "Pareto(2.5)":  "#D6604D",
    "Mixed(90/10)": "#762A83",
}

MODEL_COLORS = {
    "normal":    "#2166AC",
    "student_t": "#D6604D",
    "stable":    "#4DAC26",
}

### Load dữ liệu từ Notebook 01

In [ ]:
df = pd.read_parquet("data/generated/synthetic_distributions.parquet")
datasets = {col: df[col].to_numpy() for col in df.columns}

print(f"Loaded {len(datasets)} datasets, each n = {len(next(iter(datasets.values()))):,}")

---
## 1. Theoretical Background

### 1.1 Ba họ distribution được fit

Chúng ta fit ba **distribution** với mức độ linh hoạt **tail** tăng dần:

**Normal** (2 tham số: $\mu, \sigma$): **benchmark** baseline. Tail giảm theo $e^{-x^2/2}$ — nhanh nhất trong ba họ. Nếu Normal là model tốt nhất theo **AIC**, dữ liệu không có **heavy tail** đáng kể.

**Student-t** (3 tham số: $\nu, \mu, \sigma$): có **power-law tail** với **tail index** = $\nu$. Khi $\nu \to \infty$ thì tiệm cận **Normal**. Linh hoạt hơn **Normal** ở tail mà chỉ tốn thêm 1 tham số.

**Alpha-stable** (4 tham số: $\alpha, \beta, \mu, \sigma$): tổng quát hóa của cả **Normal** ($\alpha=2$) và **Cauchy** ($\alpha=1$). Cho phép cả **heavy tail** lẫn **asymmetry** (**skewness** $\beta \in [-1,1]$). Tốn kém nhất về tính toán.

### 1.2 Tiêu chí so sánh model

**AIC** và **BIC** cân bằng giữa **goodness-of-fit** và **model complexity**:

$$\text{AIC} = 2k - 2\ln(\hat{L}) \qquad \text{BIC} = k\ln(n) - 2\ln(\hat{L})$$

Trong đó $k$ là số **parameter**, $n$ là cỡ mẫu, $\hat{L}$ là **maximum likelihood**. Giá trị thấp hơn là tốt hơn. **BIC** phạt nặng hơn với **parameter** dư thừa khi $n$ lớn — quan trọng khi so sánh **Normal** (k=2) với **Alpha-stable** (k=4).

---
## 2. Model Fitting & Comparison

### 2.1 Fit toàn bộ dataset

Chúng ta fit cả ba **distribution** lên từng **dataset** và gom kết quả **AIC/BIC** vào một bảng tổng hợp. **Alpha-stable** tốn thời gian hơn vì cần **numerical optimization** phức tạp — đây là đánh đổi thực tế cần cân nhắc khi dùng trong production.

In [ ]:
CACHE_PATH = "../data/generated/fit_cache.pkl"

if os.path.exists(CACHE_PATH):
    fit_results = joblib.load(CACHE_PATH)
    print("Loaded from cache.")
else:
    fit_results = {}
    for ds_name, arr in datasets.items():
        print(f"Fitting: {ds_name}...")
        fit_results[ds_name] = compare_distributions(arr)
    joblib.dump(fit_results, CACHE_PATH)
    print("Done. Saved to cache.")

# Rebuild all_rows từ fit_results
all_rows = []
for ds_name, comparison in fit_results.items():
    for _, row in comparison.iterrows():
        all_rows.append({
            "Dataset":      ds_name,
            "Model":        row["distribution"],
            "Log-lik":      round(row["log_likelihood"], 1),
            "AIC":          round(row["aic"], 1),
            "BIC":          round(row["bic"], 1),
            "ΔAIC vs best": None,
        })

In [ ]:
# Tính ΔAIC so với model tốt nhất trong mỗi dataset
df_all = pd.DataFrame(all_rows)
best_aic = df_all.groupby("Dataset")["AIC"].min()
df_all["ΔAIC vs best"] = df_all.apply(
    lambda r: round(r["AIC"] - best_aic[r["Dataset"]], 1), axis=1
)

# Highlight model tốt nhất cho mỗi dataset
def highlight_best(row):
    return ["background-color: #d4edda" if row["ΔAIC vs best"] == 0.0 else "" for _ in row]

df_all.set_index(["Dataset", "Model"]).style.apply(highlight_best, axis=1)

**ΔAIC** cho biết model kém hơn bao nhiêu so với model tốt nhất trong cùng **dataset**. Quy tắc thực hành: ΔAIC < 2 là tương đương, 2–10 là model yếu hơn đáng kể, >10 là model rõ ràng kém hơn.

### 2.2 Best model theo từng dataset

In [ ]:
print("Best model (by AIC) for each dataset:\n")
for ds_name, arr in datasets.items():
    best = best_distribution(arr, criterion="aic")
    print(f"  {ds_name:18s}  →  {best['distribution']:10s}  "
          f"AIC={best['aic']:.1f}")

---
## 3. Fitted Parameters

Ngoài **AIC/BIC**, các **parameter** của model fit cũng mang thông tin định lượng quan trọng. Đặc biệt:
- **Student-t**: $\hat{\nu}$ (df) ước lượng **tail index** — $\hat{\nu}$ nhỏ = **tail** nặng hơn.
- **Alpha-stable**: $\hat{\alpha}$ (stability index) — $\hat{\alpha} < 2$ xác nhận **heavy tail**, $\hat{\beta}$ đo **asymmetry**.

In [ ]:
param_rows = []
for ds_name, arr in datasets.items():
    # Student-t params
    t_fit = fit_distribution(arr, "student_t")
    # Stable params
    s_fit = fit_distribution(arr, "stable")

    param_rows.append({
        "Dataset":         ds_name,
        # Student-t
        "t: df (ν̂)":      round(t_fit["df"], 2),
        "t: loc":          round(t_fit["loc"], 4),
        "t: scale":        round(t_fit["scale"], 4),
        # Alpha-stable
        "stable: α":       round(s_fit["alpha"], 3),
        "stable: β":       round(s_fit["beta"], 3),
        "stable: scale":   round(s_fit["scale"], 4),
    })

df_params = pd.DataFrame(param_rows).set_index("Dataset")
df_params

Quan sát quan trọng:
- **Student-t df** ước lượng đúng thứ tự **tail heaviness**: **Pareto** và **Student-t(3)** cho $\hat{\nu}$ nhỏ nhất, **Gaussian** cho $\hat{\nu}$ rất lớn (tiệm cận **Normal**).
- **Alpha-stable α** < 2 với tất cả **heavy-tail dataset** — xác nhận **sub-Gaussian tail**. Với **Gaussian** data, $\hat{\alpha}$ gần 2.0 đúng như lý thuyết.
- **stable β** gần 0 với **symmetric distribution** (Gaussian, Student-t), khác 0 với **Pareto** — phản ánh **right-skewness** của Pareto sau khi demean.

---
## 4. Goodness-of-Fit Assessment

**AIC/BIC** đo chất lượng fit toàn bộ **distribution**. Nhưng trong **fat-tail modeling**, chúng ta quan tâm đặc biệt đến chất lượng fit ở **tail** — vì đó là vùng ảnh hưởng trực tiếp đến **risk metric** như **VaR** và **ES**. Chúng ta dùng hai loại đánh giá bổ sung:

- **KS statistic**: khoảng cách tối đa giữa **empirical CDF** và **fitted CDF** — đo lỗi ở toàn bộ **distribution**.
- **Tail quantile error**: lỗi ở các **quantile** cực trị (Q95 đến Q99.9) — đo lỗi tập trung vào **tail**.

### 4.1 KS Distance

In [ ]:
ks_rows = []
for ds_name, arr in datasets.items():
    row = {"Dataset": ds_name}
    # Normal
    n_fit = fit_distribution(arr, "normal")
    row["KS: Normal"] = round(
        kolmogorov_smirnov_distance(
            arr, lambda x: stats.norm.cdf(x, loc=n_fit["mu"], scale=n_fit["sigma"])
        ), 4
    )
    # Student-t
    t_fit = fit_distribution(arr, "student_t")
    row["KS: Student-t"] = round(
        kolmogorov_smirnov_distance(
            arr, lambda x: stats.t.cdf(x, df=t_fit["df"], loc=t_fit["loc"], scale=t_fit["scale"])
        ), 4
    )
    # Stable
    s_fit = fit_distribution(arr, "stable")
    row["KS: Stable"] = round(
        kolmogorov_smirnov_distance(
            arr, lambda x: stats.levy_stable.cdf(
                x, alpha=s_fit["alpha"], beta=s_fit["beta"],
                loc=s_fit["loc"], scale=s_fit["scale"]
            )
        ), 4
    )
    ks_rows.append(row)

df_ks = pd.DataFrame(ks_rows).set_index("Dataset")
# Highlight minimum KS trong mỗi row
df_ks.style.highlight_min(axis=1, color="#d4edda")

### 4.2 Tail Quantile Error

**KS statistic** nhạy nhất ở vùng giữa **distribution** (nơi **CDF** thay đổi nhanh nhất), không phải ở **tail**. Để đánh giá chất lượng fit ở **tail**, chúng ta tính **mean absolute error** giữa **empirical quantile** và **model quantile** ở các mức xác suất cực trị Q95–Q99.9.

In [ ]:
tail_probs = np.array([0.95, 0.975, 0.99, 0.995, 0.999])
tail_rows  = []

for ds_name, arr in datasets.items():
    row = {"Dataset": ds_name}

    # Normal
    n_fit = fit_distribution(arr, "normal")
    err_n = tail_quantile_error(
        arr,
        lambda p: stats.norm.ppf(p, loc=n_fit["mu"], scale=n_fit["sigma"]),
        tail_probs,
    )
    row["MAE tail: Normal"] = round(err_n["mean_absolute_error"], 4)

    # Student-t
    t_fit = fit_distribution(arr, "student_t")
    err_t = tail_quantile_error(
        arr,
        lambda p: stats.t.ppf(p, df=t_fit["df"], loc=t_fit["loc"], scale=t_fit["scale"]),
        tail_probs,
    )
    row["MAE tail: Student-t"] = round(err_t["mean_absolute_error"], 4)

    # Stable
    s_fit = fit_distribution(arr, "stable")
    err_s = tail_quantile_error(
        arr,
        lambda p: stats.levy_stable.ppf(
            p, alpha=s_fit["alpha"], beta=s_fit["beta"],
            loc=s_fit["loc"], scale=s_fit["scale"]
        ),
        tail_probs,
    )
    row["MAE tail: Stable"] = round(err_s["mean_absolute_error"], 4)

    tail_rows.append(row)

df_tail_err = pd.DataFrame(tail_rows).set_index("Dataset")
df_tail_err.style.highlight_min(axis=1, color="#d4edda")

Kết quả hai bảng thường không giống nhau hoàn toàn: model có **KS** nhỏ nhất chưa chắc có **tail quantile error** nhỏ nhất. Đây là lý do tại sao trong **tail risk modeling**, việc đánh giá chất lượng fit phải được thực hiện riêng biệt ở **tail region** — không thể dùng chỉ một **metric** duy nhất.

---
## 5. Visual Diagnostics

### 5.1 Fitted PDF vs. Empirical Density

Chúng ta vẽ **fitted PDF** của cả ba model chồng lên **KDE** thực nghiệm. Panel trái cho thấy toàn bộ **distribution**, panel phải zoom vào **right tail** với **log-scale** trên trục y để thấy rõ sự khác biệt ở vùng **extreme**.

In [ ]:
n_datasets = len(datasets)
fig, axes  = plt.subplots(n_datasets, 2, figsize=(13, 3.5 * n_datasets))

for row_idx, (ds_name, arr) in enumerate(datasets.items()):
    # Fit tất cả models
    n_fit = fit_distribution(arr, "normal")
    t_fit = fit_distribution(arr, "student_t")
    s_fit = fit_distribution(arr, "stable")

    x_lo, x_hi = np.quantile(arr, 0.001), np.quantile(arr, 0.999)
    x_full = np.linspace(x_lo, x_hi, 400)
    x_tail = np.linspace(np.quantile(arr, 0.80), x_hi, 300)

    pdf_normal = stats.norm.pdf(x_full, loc=n_fit["mu"], scale=n_fit["sigma"])
    pdf_t      = stats.t.pdf(x_full, df=t_fit["df"], loc=t_fit["loc"], scale=t_fit["scale"])
    pdf_stable = stats.levy_stable.pdf(
        x_full, alpha=s_fit["alpha"], beta=s_fit["beta"],
        loc=s_fit["loc"], scale=s_fit["scale"]
    )

    kde = stats.gaussian_kde(arr, bw_method=0.25)

    for col_idx, (x_grid, ax, title_sfx) in enumerate([
        (x_full, axes[row_idx, 0], "Full"),
        (x_tail, axes[row_idx, 1], "Right Tail [log]"),
    ]):
        ax.fill_between(x_grid, kde(x_grid), alpha=0.15,
                        color=COLORS[ds_name], label="Empirical KDE")
        ax.plot(x_grid, kde(x_grid),
                color=COLORS[ds_name], lw=1.5, alpha=0.8)
        ax.plot(x_grid,
                stats.norm.pdf(x_grid, loc=n_fit["mu"], scale=n_fit["sigma"]),
                color=MODEL_COLORS["normal"], lw=1.8, ls="-", label="Normal")
        ax.plot(x_grid,
                stats.t.pdf(x_grid, df=t_fit["df"], loc=t_fit["loc"], scale=t_fit["scale"]),
                color=MODEL_COLORS["student_t"], lw=1.8, ls="--", label="Student-t")
        ax.plot(x_grid,
                stats.levy_stable.pdf(
                    x_grid, alpha=s_fit["alpha"], beta=s_fit["beta"],
                    loc=s_fit["loc"], scale=s_fit["scale"]
                ),
                color=MODEL_COLORS["stable"], lw=1.8, ls=":", label="Stable")

        if col_idx == 1:
            ax.set_yscale("log")
        ax.set_title(f"{ds_name} — {title_sfx}", fontsize=10)
        ax.set_xlabel("Value", fontsize=9)
        ax.set_ylabel("Density", fontsize=9)
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

fig.suptitle("Fitted PDF vs. Empirical KDE  (left: full · right: tail log-scale)",
             fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

Panel **right tail log-scale** là nơi thông tin quan trọng nhất: model nào khớp tốt với **empirical density** ở vùng $x > Q_{80}$? Với **Pareto** và **Student-t(3)**, **Normal** (đường liền) thường tụt dốc nhanh hơn **empirical** — đây là **tail underestimation** điển hình của **Gaussian model**.

### 5.2 QQ Plot — Tail Focus

**QQ plot** tập trung vào **upper tail** (top 10%) để so sánh **empirical quantile** với **model quantile** trực tiếp. Điểm nằm trên đường 45° = model **underestimate** **tail quantile** (nguy hiểm trong **risk management**).

In [ ]:
tail_frac  = 0.10   # zoom vào top 10%
n_datasets = len(datasets)

fig, axes = plt.subplots(n_datasets, 3, figsize=(13, 3.2 * n_datasets))

model_specs = [
    ("Normal",    "normal",    MODEL_COLORS["normal"]),
    ("Student-t", "student_t", MODEL_COLORS["student_t"]),
    ("Stable",    "stable",    MODEL_COLORS["stable"]),
]

for row_idx, (ds_name, arr) in enumerate(datasets.items()):
    n      = len(arr)
    n_tail = max(5, int(n * tail_frac))
    tail_p = np.linspace(1 - tail_frac, 1 - 1 / n, n_tail)
    emp_q  = np.quantile(arr, tail_p)

    for col_idx, (model_label, dist_key, color) in enumerate(model_specs):
        ax     = axes[row_idx, col_idx]
        fit    = fit_distribution(arr, dist_key)

        if dist_key == "normal":
            theo_q = stats.norm.ppf(tail_p, loc=fit["mu"], scale=fit["sigma"])
        elif dist_key == "student_t":
            theo_q = stats.t.ppf(tail_p, df=fit["df"], loc=fit["loc"], scale=fit["scale"])
        else:
            theo_q = stats.levy_stable.ppf(
                tail_p, alpha=fit["alpha"], beta=fit["beta"],
                loc=fit["loc"], scale=fit["scale"]
            )

        ax.scatter(theo_q, emp_q, s=15, alpha=0.7, color=color)
        lim = np.nanmax(np.abs(np.concatenate([theo_q, emp_q])))
        ax.plot([-lim, lim], [-lim, lim], "r--", lw=1.2, alpha=0.7)
        ax.set_title(f"{ds_name}\n{model_label}", fontsize=9)
        ax.set_xlabel("Model quantile", fontsize=8)
        ax.set_ylabel("Empirical quantile", fontsize=8)
        ax.grid(True, alpha=0.3)

fig.suptitle(f"Tail QQ Plot (upper {int(tail_frac*100)}%)  —  điểm trên đường = model underestimates tail",
             fontsize=11, y=1.005)
plt.tight_layout()
plt.show()

---
## 6. Bootstrap Confidence Intervals cho Parameters

**MLE parameter estimate** là **point estimate** — chưa cho biết mức độ không chắc chắn. Với **bootstrap**, chúng ta lấy mẫu lại từ dữ liệu gốc nhiều lần, fit lại mỗi lần, và tính **CI** từ phân phối của các **estimate**.

Chúng ta tập trung vào **Student-t df** ($\hat{\nu}$) và **Alpha-stable α** ($\hat{\alpha}$) — hai **parameter** trực tiếp định lượng **tail heaviness** — vì **estimation uncertainty** ở đây ảnh hưởng trực tiếp đến **risk metric**.

In [ ]:
N_BOOTSTRAP = 300   # giảm để chạy nhanh; dùng 500+ cho kết quả chính xác hơn
SEED        = 42

ci_rows = []
for ds_name, arr in datasets.items():
    print(f"Bootstrap CI: {ds_name}...")

    # CI cho Student-t df
    t_cis = bootstrap_parameter_cis(
        arr, student_t.fit,
        n_bootstrap=N_BOOTSTRAP, confidence_level=0.95, seed=SEED
    )
    # CI cho Stable alpha
    s_cis = bootstrap_parameter_cis(
        arr, stable.fit,
        n_bootstrap=N_BOOTSTRAP, confidence_level=0.95, seed=SEED
    )

    ci_rows.append({
        "Dataset":             ds_name,
        "t-df  point est":     round(t_cis["df"]["estimate"], 2),
        "t-df  95% CI lower": round(t_cis["df"]["ci_lower"], 2),
        "t-df  95% CI upper": round(t_cis["df"]["ci_upper"], 2),
        "t-df  SE":            round(t_cis["df"]["se"], 3),
        "stable-α point est": round(s_cis["alpha"]["estimate"], 3),
        "stable-α CI lower":  round(s_cis["alpha"]["ci_lower"], 3),
        "stable-α CI upper":  round(s_cis["alpha"]["ci_upper"], 3),
        "stable-α SE":        round(s_cis["alpha"]["se"], 4),
    })

print("\nDone.")
df_ci = pd.DataFrame(ci_rows).set_index("Dataset")
df_ci

In [ ]:
# Trực quan hóa bootstrap CI cho Student-t df
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ds_names   = df_ci.index.tolist()
y_pos      = np.arange(len(ds_names))

# Panel trái: Student-t df
ax = axes[0]
for i, name in enumerate(ds_names):
    est = df_ci.loc[name, "t-df  point est"]
    lo  = df_ci.loc[name, "t-df  95% CI lower"]
    hi  = df_ci.loc[name, "t-df  95% CI upper"]
    ax.plot([lo, hi], [i, i], lw=3, color=COLORS[name], alpha=0.7)
    ax.scatter(est, i, s=80, zorder=5, color=COLORS[name])
ax.set_yticks(y_pos)
ax.set_yticklabels(ds_names, fontsize=9)
ax.set_xlabel("Student-t  df  (ν̂)")
ax.set_title("Bootstrap 95% CI — Student-t df\n(nhỏ hơn = heavy tail hơn)")
ax.axvline(30, color="gray", ls="--", lw=1, alpha=0.5, label="df=30 ≈ Gaussian")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="x")

# Panel phải: Stable alpha
ax = axes[1]
for i, name in enumerate(ds_names):
    est = df_ci.loc[name, "stable-α point est"]
    lo  = df_ci.loc[name, "stable-α CI lower"]
    hi  = df_ci.loc[name, "stable-α CI upper"]
    ax.plot([lo, hi], [i, i], lw=3, color=COLORS[name], alpha=0.7)
    ax.scatter(est, i, s=80, zorder=5, color=COLORS[name])
ax.set_yticks(y_pos)
ax.set_yticklabels(ds_names, fontsize=9)
ax.set_xlabel("Alpha-stable  α")
ax.set_title("Bootstrap 95% CI — Stable α\n(< 2 = heavy tail · = 2 = Gaussian)")
ax.axvline(2.0, color="gray", ls="--", lw=1, alpha=0.5, label="α=2 (Gaussian)")
ax.set_xlim(0.8, 2.2)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="x")

fig.suptitle("Parameter Estimation Uncertainty — 95% Bootstrap CI", fontsize=12)
plt.tight_layout()
plt.show()

**CI** rộng = **estimation uncertainty** cao. Với **Student-t df**, các **dataset** có **heavy tail** thực sự cho **CI** hẹp và tập trung ở giá trị thấp — **estimator** hoạt động tốt. Với **Gaussian** data, **CI** của df rất rộng và trải ra cao — đúng vì **Student-t** với df lớn gần như không phân biệt được nhau. Với **stable α**, **CI** không chứa 2.0 với các **heavy-tail dataset** — xác nhận thống kê rằng **Gaussian assumption** bị vi phạm.

---
## 7. Summary

Trong notebook này chúng ta đã fit và đánh giá ba họ **parametric distribution** trên năm **dataset**:

1. **Model selection qua AIC/BIC**: **Student-t** và **Stable** thắng **Normal** trên tất cả **heavy-tail dataset** với **ΔAIC** lớn — xác nhận **Normal assumption** không phù hợp khi có **heavy tail**.
2. **Fitted parameters**: **Student-t df** ($\hat{\nu}$) và **stable α** ($\hat{\alpha}$) ước lượng đúng thứ tự **tail heaviness** theo lý thuyết.
3. **Goodness-of-fit**: **KS distance** và **tail quantile MAE** bổ sung cho **AIC/BIC** — model tốt toàn bộ chưa chắc tốt ở **tail**, và ngược lại.
4. **Visual diagnostics**: **log-scale tail PDF** và **tail QQ plot** cho thấy rõ **Normal** underestimate **tail quantile** — đặc biệt nghiêm trọng với **Pareto** và **Student-t(3)**.
5. **Bootstrap CI**: **estimation uncertainty** của **tail parameter** ($\hat{\nu}$, $\hat{\alpha}$) định lượng được bằng **bootstrap** — thông tin quan trọng trước khi dùng **parameter** để tính **risk metric**.

**Next:** `04_tail_behavior_analysis.ipynb` — phân tích sâu hơn vào vùng **extreme** để định lượng **tail behavior** vượt ra ngoài các **summary statistic** thông thường.